# Drawing resource call graphs

A call graph records declared component dependencies and their multiplicities.
Compiler builds the graph and evaluates costs. Drawer formats the graph without
constructing or executing a quantum circuit. This native example follows the
locked call-graph notebook with explicit resource declarations.


In [ ]:
from IPython.display import SVG, display
from examples.algorithms.resource_call_graphs import declared_example
from flagquantum.compiler._component_trace.call_graph import symbolic_call_graph
from flagquantum.compiler._component_trace.resource_model import ComponentCost
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg

root, declarations = declared_example()
call_graph = symbolic_call_graph(root, declarations)

def node_title(node):
    return node.name if isinstance(node, ComponentCost) else str(node)

plain_dot = export_call_graph_dot(root, call_graph.direct_calls, title=node_title)
renders = [render_call_graph_svg(plain_dot)]
display(SVG(renders[-1]))


## DOT and SVG export

The exported DOT is editable text. The renderer returns SVG bytes that can be
written to a caller-owned file. Rendering requires a local Graphviz executable.


In [ ]:
svg = render_call_graph_svg(plain_dot)
renders.append(svg)
print(svg[:400], "...")


## Annotating declared costs

The existing recursive Compiler query supplies each node's full symbolic cost.
The atomic declaration carries 100 T gates. The root's model is 100*n + 600 T
gates, with zero declared Clifford gates and rotations. These synthetic model
counts describe the demonstration graph. No primitive instruction count or
quantum work-wire allocation is measured by drawing it.


In [ ]:
from examples.algorithms.resource_call_graphs import annotated_example

annotated_graph, annotations = annotated_example()
annotated_dot = export_call_graph_dot(
    annotated_graph.root,
    annotated_graph.direct_calls,
    title=node_title,
    annotations=annotations,
)
renders.append(render_call_graph_svg(annotated_dot))
display(SVG(renders[-1]))
assert annotations[annotated_graph.root]["T count"] == "100*_n0 + 600"


The existing title callback and annotation mapping provide presentation
customization. Resource keys retain node identity when display labels coincide.
Compiler owns graph traversal and costs, while Drawer owns DOT and SVG lifecycle.
Graph construction adds zero quantum gates, work wires or measurements.
